# DPO Training — Real Run

**Purpose:** train Qwen 2.5 1.5B-Instruct on the FormatBench training set using DPO + LoRA. This is the main training run.

**Carries over from the smoke test:** bfloat16 model, LoRA adapters (so it fits on a T4), gradient checkpointing, 8-bit Adam optimizer.

**What's new vs smoke test:**
- Full training split (~440 examples), not 2
- Real number of training steps (1 epoch)
- Validation during training
- Saves the trained LoRA adapter
- Compares model output before vs after training on sample prompts

**Expected runtime:** ~30-50 minutes on a Kaggle T4.

**Kaggle setup:**
1. Create new Kaggle Notebook
2. Settings → Accelerator → **GPU T4 x2** (or any GPU)
3. Settings → Internet → **On**
4. Paste this notebook in, edit `HF_USERNAME`, run all cells

**Important:** evaluation of the trained model on the test set + adversarial set happens in the *next* notebook (`dpo_03_evaluate.ipynb`). This notebook produces the trained adapter; the next one tells us whether it actually learned something useful.

---
*Part of the [Prosify project](https://github.com/[your-username]/prosify).*

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers trl datasets accelerate peft bitsandbytes torchao

import transformers, trl, datasets, accelerate, torch, peft, bitsandbytes, torchao
print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"peft:         {peft.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"torchao:      {torchao.__version__}")
print(f"torch:        {torch.__version__}")

## 2. Configuration

Set your HuggingFace username. The hyperparameters below are conservative starting points based on the reward-hacking risk we identified in the baseline classifier experiment (100% test / 1.2% adversarial — see the Prosify project README).

In [ ]:
# ===== EDIT THIS =====
HF_USERNAME = "your-hf-username"  # <-- your HuggingFace username
# =====================

DATASET_REPO = f"{HF_USERNAME}/formatbench"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./prosify_qwen_1.5b_lora"

# Training hyperparameters
SEED = 42
NUM_EPOCHS = 1                   # start with 1; more epochs increases reward-hacking risk
PER_DEVICE_BATCH = 1             # T4 memory constraint
GRADIENT_ACCUMULATION = 4        # effective batch size = 4
LEARNING_RATE = 5e-5             # higher than full fine-tune because LoRA params are small
WARMUP_RATIO = 0.1
MAX_LENGTH = 1024
BETA = 0.1                       # DPO KL-leash strength (lower = looser, higher = tighter)

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f"Will train for {NUM_EPOCHS} epoch(s)")
print(f"Effective batch size: {PER_DEVICE_BATCH * GRADIENT_ACCUMULATION}")
print(f"Output dir: {OUTPUT_DIR}")

## 3. Confirm GPU

In [ ]:
import torch, random, numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory:     {mem_gb:.1f} GB")
else:
    raise RuntimeError("No GPU detected.")

## 4. Load base model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {BASE_MODEL}...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="auto",
)

model.config.use_cache = False  # required for gradient checkpointing

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nModel loaded.")
print(f"  Parameters:     {n_params:.2f}B")
print(f"  GPU mem in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Capture base-model responses on sample prompts (BEFORE training)

We'll generate responses to a few representative prompts now, before any training. After training, we'll generate again on the same prompts to see how the trained model differs from the base.

In [ ]:
# A few representative prompts spanning major dataset categories.
# These should produce prose responses if training works.
SAMPLE_PROMPTS = [
    "Write an email to my manager Alex saying I'll be working from home tomorrow because the plumber is coming.",
    "How does intermittent fasting actually work? Explain it simply.",
    "My friend just told me she's anxious about her interview tomorrow. Help me message her something supportive.",
]

def generate(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

base_responses = []
for i, prompt in enumerate(SAMPLE_PROMPTS, 1):
    print(f"=== PROMPT {i} (BASE MODEL, BEFORE TRAINING) ===")
    print(f"User: {prompt}\n")
    response = generate(prompt)
    base_responses.append(response)
    print(f"Response:\n{response}\n")
    print("=" * 60 + "\n")

## 6. Load the FormatBench dataset and recreate splits

Same seed (42) and same stratified-by-context logic as the baseline classifier notebook, so the splits are identical.

In [ ]:
from datasets import load_dataset, Dataset
from collections import defaultdict

raw = load_dataset(DATASET_REPO, split="train")
print(f"Loaded {len(raw)} preference pairs")

rng = random.Random(SEED)
rows = list(raw)

by_context = defaultdict(list)
for row in rows:
    by_context[row["context"]].append(row)

train_rows, val_rows, test_rows = [], [], []
for context, items in sorted(by_context.items()):
    shuffled = items[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, n // 10)
    n_val = max(1, n // 10)
    n_train = n - n_test - n_val
    train_rows.extend(shuffled[:n_train])
    val_rows.extend(shuffled[n_train:n_train + n_val])
    test_rows.extend(shuffled[n_train + n_val:])

rng.shuffle(train_rows)
rng.shuffle(val_rows)
rng.shuffle(test_rows)

print(f"Train: {len(train_rows)}  |  Val: {len(val_rows)}  |  Test: {len(test_rows)}")

## 7. Apply chat template formatting

In [ ]:
def format_with_chat_template(example):
    formatted_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

# Wrap into HF Datasets for the trainer
train_ds = Dataset.from_list(train_rows).map(
    format_with_chat_template,
    remove_columns=[c for c in Dataset.from_list(train_rows).column_names if c not in ["prompt", "chosen", "rejected"]],
)
val_ds = Dataset.from_list(val_rows).map(
    format_with_chat_template,
    remove_columns=[c for c in Dataset.from_list(val_rows).column_names if c not in ["prompt", "chosen", "rejected"]],
)

print(f"Train dataset: {len(train_ds)} examples, columns: {train_ds.column_names}")
print(f"Val dataset:   {len(val_ds)} examples")

## 8. Configure LoRA and the DPOTrainer

In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

train_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    beta=BETA,
    max_length=MAX_LENGTH,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    seed=SEED,
)

trainer = DPOTrainer(
    model=model,
    args=train_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"\n✓ DPOTrainer ready")
print(f"  Trainable params: {trainable / 1e6:.2f}M / {total / 1e9:.2f}B ({100*trainable/total:.2f}%)")
print(f"  Total training steps: ~{(len(train_ds) // (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION)) * NUM_EPOCHS}")

## 9. Train

Expect ~30-50 minutes on a T4. The trainer will print loss every 10 steps and evaluate at the end of the epoch.

In [ ]:
trainer.train()

## 10. Save the LoRA adapter

We save just the LoRA adapter weights (small — a few MB), not the entire base model. To use the trained model later, you load the base model + apply the adapter.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
saved_files = os.listdir(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
print(f"Files: {sorted(saved_files)}")

## 11. Generate from the trained model on the same sample prompts

Same prompts as section 5. Side-by-side comparison shows whether training meaningfully changed the model's behavior.

In [ ]:
# trainer.model is now the trained (LoRA-adapted) model
trained_model = trainer.model
trained_model.eval()

def generate_with(target_model, prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(target_model.device)
    with torch.no_grad():
        output = target_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for i, prompt in enumerate(SAMPLE_PROMPTS):
    print(f"=== PROMPT {i+1} ===")
    print(f"User: {prompt}\n")
    print("--- BASE MODEL (before training) ---")
    print(base_responses[i])
    print("\n--- TRAINED MODEL (after DPO) ---")
    trained_response = generate_with(trained_model, prompt)
    print(trained_response)
    print("\n" + "=" * 60 + "\n")

## 12. (Optional) Push the LoRA adapter to Hugging Face

If you want the trained adapter publicly available — and you do, eventually — push it to HuggingFace. Uncomment the cells below.

This pushes just the adapter (a few MB), not the full model. Anyone can then load the base model + apply your adapter to get the trained version.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()  # paste your HF write token when prompted

In [ ]:
# ADAPTER_REPO = f"{HF_USERNAME}/prosify-qwen-1.5b-lora"
# trainer.push_to_hub(ADAPTER_REPO)
# print(f"Pushed to https://huggingface.co/{ADAPTER_REPO}")

## 13. Summary

If you got here without errors, you have:

✅ A DPO-trained LoRA adapter saved to `OUTPUT_DIR`  
✅ Before/after comparisons on sample prompts (eyeball check that training did *something*)  
✅ Training and eval loss curves printed during training  

**What we don't know yet:** whether the trained model actually solves the problem — i.e., whether it removes over-formatting while preserving structure where structure helps (the 1.2% → ??? on adversarial gap).

That's what `dpo_03_evaluate.ipynb` will measure:
- Win rate on the held-out test split
- Accuracy on the adversarial held-out set
- Per-category breakdown

If both numbers are good, the project has its headline result. If the trained model wins on the main test set but fails on adversarial — same shortcut the classifier learned — we'll need to adjust (higher `beta`, fewer epochs, or different LoRA config) and try again.